# OSMI Mental Health in Tech (2016): Data Cleaning

This is a rework of my original Project 1 analysis. The first version used `WHERE IS NOT NULL` filters in SQL without actually checking what those nulls meant or how much data they represented. 
This notebook fixes that: I'm inspecting each issue in the raw data, deciding how to handle it, and 
documenting why.

Raw dataset: OSMI Mental Health in Tech 2016 survey, 1,433 responses, 63 columns.

In [1]:
import pandas as pd

df_raw = pd.read_csv('mental-heath-in-tech-2016_20161114.csv')
print(df_raw.shape)
df_raw.columns.tolist()

(1433, 63)


['Are you self-employed?',
 'How many employees does your company or organization have?',
 'Is your employer primarily a tech company/organization?',
 'Is your primary role within your company related to tech/IT?',
 'Does your employer provide mental health benefits as part of healthcare coverage?',
 'Do you know the options for mental health care available under your employer-provided coverage?',
 'Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?',
 'Does your employer offer resources to learn more about mental health concerns and options for seeking help?',
 'Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?',
 'If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:',
 'Do you think that discussing a mental health disorder with your employer would have neg

## Translation layer: mapping clean names to the original survey questions

Raw survey questions make poor column names in working code — long, spaced, full of
punctuation. But the exact question wording is real data in a psychology survey; losing it
loses methodological context. This dictionary keeps both: clean snake_case names for every
line of code below, with the exact original question preserved and accessible whenever it's
needed (e.g. for chart titles later).

In [2]:
question_map = {
    'self_employed': 'Are you self-employed?',
    'num_employees': 'How many employees does your company or organization have?',
    'employer_is_tech_company': 'Is your employer primarily a tech company/organization?',
    'role_is_tech_related': 'Is your primary role within your company related to tech/IT?',
    'employer_provides_mh_benefits': 'Does your employer provide mental health benefits as part of healthcare coverage?',
    'know_mh_options_current': 'Do you know the options for mental health care available under your employer-provided coverage?',
    'employer_formal_mh_discussion_current': 'Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?',
    'employer_offers_mh_resources_current': 'Does your employer offer resources to learn more about mental health concerns and options for seeking help?',
    'mh_anonymity_protected_current': 'Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?',
    'mh_medical_leave_difficulty': 'If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:',
    'mh_employer_consequences_current': 'Do you think that discussing a mental health disorder with your employer would have negative consequences?',
    'physical_health_employer_consequences_current': 'Do you think that discussing a physical health issue with your employer would have negative consequences?',
    'comfortable_coworkers_mh_current': 'Would you feel comfortable discussing a mental health disorder with your coworkers?',
    'comfortable_supervisor_mh_current': 'Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?',
    'employer_takes_mh_seriously_current': 'Do you feel that your employer takes mental health as seriously as physical health?',
    'observed_mh_negative_consequences_current': 'Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?',
    'has_mh_medical_coverage': 'Do you have medical coverage (private insurance or state-provided) which includes treatment of \xa0mental health issues?',
    'knows_mh_local_resources': 'Do you know local or online resources to seek help for a mental health disorder?',
    'reveals_mh_to_clients': 'If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?',
    'reveal_mh_to_client_negative_impact': 'If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?',
    'reveals_mh_to_coworkers': 'If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?',
    'reveal_mh_to_coworker_negative_impact': 'If you have revealed a mental health issue to a coworker or employee, do you believe this has impacted you negatively?',
    'mh_productivity_affected_selfemployed': 'Do you believe your productivity is ever affected by a mental health issue?',
    'pct_worktime_mh_affected_selfemployed': 'If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?',
    'has_previous_employers': 'Do you have previous employers?',
    'employer_provides_mh_benefits_previous': 'Have your previous employers provided mental health benefits?',
    'know_mh_options_previous': 'Were you aware of the options for mental health care provided by your previous employers?',
    'employer_formal_mh_discussion_previous': 'Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?',
    'employer_offers_mh_resources_previous': 'Did your previous employers provide resources to learn more about mental health issues and how to seek help?',
    'mh_anonymity_protected_previous': 'Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?',
    'mh_employer_consequences_previous': 'Do you think that discussing a mental health disorder with previous employers would have negative consequences?',
    'physical_health_employer_consequences_previous': 'Do you think that discussing a physical health issue with previous employers would have negative consequences?',
    'willing_mh_coworkers_previous': 'Would you have been willing to discuss a mental health issue with your previous co-workers?',
    'willing_mh_supervisor_previous': 'Would you have been willing to discuss a mental health issue with your direct supervisor(s)?',
    'employer_took_mh_seriously_previous': 'Did you feel that your previous employers took mental health as seriously as physical health?',
    'observed_mh_negative_consequences_previous': 'Did you hear of or observe negative consequences for co-workers with mental health issues in your previous workplaces?',
    'willing_physical_health_interview': 'Would you be willing to bring up a physical health issue with a potential employer in an interview?',
    'why_or_why_not_physical': 'Why or why not?',
    'willing_mh_interview': 'Would you bring up a mental health issue with a potential employer in an interview?',
    'why_or_why_not_mh': 'Why or why not?.1',
    'identified_mh_would_hurt_career': 'Do you feel that being identified as a person with a mental health issue would hurt your career?',
    'coworkers_view_mh_negatively': 'Do you think that team members/co-workers would view you more negatively if they knew you suffered from a mental health issue?',
    'willing_share_mh_friends_family': 'How willing would you be to share with friends and family that you have a mental illness?',
    'observed_mh_unsupportive_response': 'Have you observed or experienced an unsupportive or badly handled response to a mental health issue in your current or previous workplace?',
    'observation_reduced_own_mh_disclosure': 'Have your observations of how another individual who discussed a mental health disorder made you less likely to reveal a mental health issue yourself in your current workplace?',
    'family_mh_history': 'Do you have a family history of mental illness?',
    'had_mh_disorder_past': 'Have you had a mental health disorder in the past?',
    'has_mh_disorder_current': 'Do you currently have a mental health disorder?',
    'mh_conditions_diagnosed_yes': 'If yes, what condition(s) have you been diagnosed with?',
    'mh_conditions_believed_maybe': 'If maybe, what condition(s) do you believe you have?',
    'diagnosed_mh_by_professional': 'Have you been diagnosed with a mental health condition by a medical professional?',
    'mh_conditions_diagnosed_professional': 'If so, what condition(s) were you diagnosed with?',
    'sought_mh_treatment': 'Have you ever sought treatment for a mental health issue from a mental health professional?',
    'mh_interferes_work_when_treated': 'If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?',
    'mh_interferes_work_when_untreated': 'If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?',
    'age': 'What is your age?',
    'gender': 'What is your gender?',
    'country_live': 'What country do you live in?',
    'us_state_live': 'What US state or territory do you live in?',
    'country_work': 'What country do you work in?',
    'us_state_work': 'What US state or territory do you work in?',
    'work_position': 'Which of the following best describes your work position?',
    'works_remotely': 'Do you work remotely?',
}

rename_map = {v: k for k, v in question_map.items()}
df = df_raw.rename(columns=rename_map)
print(df.columns.tolist())
print(df.shape)


['self_employed', 'num_employees', 'employer_is_tech_company', 'role_is_tech_related', 'employer_provides_mh_benefits', 'know_mh_options_current', 'employer_formal_mh_discussion_current', 'employer_offers_mh_resources_current', 'mh_anonymity_protected_current', 'mh_medical_leave_difficulty', 'mh_employer_consequences_current', 'physical_health_employer_consequences_current', 'comfortable_coworkers_mh_current', 'comfortable_supervisor_mh_current', 'employer_takes_mh_seriously_current', 'observed_mh_negative_consequences_current', 'has_mh_medical_coverage', 'knows_mh_local_resources', 'reveals_mh_to_clients', 'reveal_mh_to_client_negative_impact', 'reveals_mh_to_coworkers', 'reveal_mh_to_coworker_negative_impact', 'mh_productivity_affected_selfemployed', 'pct_worktime_mh_affected_selfemployed', 'has_previous_employers', 'employer_provides_mh_benefits_previous', 'know_mh_options_previous', 'employer_formal_mh_discussion_previous', 'employer_offers_mh_resources_previous', 'mh_anonymity_pro

## Missingness isn't random, it's survey branching logic

Before dropping or filling in anything, I checked *why* values were missing. This survey uses skip 
logic: if you say you're self-employed, you never see the employer-benefits questions. If you say 
you have no previous employers, you skip that whole block.

That means high missingness in a column isn't automatically a data quality problem. Some columns 
show 80-90% missing simply because most respondents' answers to an earlier question meant the 
follow-up didn't apply to them.

Decision: conditional columns like these are left untouched. NaN here means "not applicable," not 
"unknown." I only cleaned the columns that actually feed my four analysis queries (age, gender, 
remote work, treatment history, disorder status, supervisor comfort), where missingness was close 
to 0% and any gaps are real issues, not survey structure.

In [3]:
missing_pct = df.isna().mean().mul(100).round(1).sort_values(ascending=False)
missing_pct.head(15)

reveal_mh_to_client_negative_impact      90.0
pct_worktime_mh_affected_selfemployed    85.8
role_is_tech_related                     81.6
has_mh_medical_coverage                  80.0
reveals_mh_to_clients                    80.0
reveals_mh_to_coworkers                  80.0
reveal_mh_to_coworker_negative_impact    80.0
knows_mh_local_resources                 80.0
mh_productivity_affected_selfemployed    80.0
mh_conditions_believed_maybe             77.5
mh_conditions_diagnosed_yes              60.4
observation_reduced_own_mh_disclosure    54.2
mh_conditions_diagnosed_professional     50.4
us_state_live                            41.4
us_state_work                            40.6
dtype: float64

In [4]:
analysis_cols = [
    'age',
    'gender',
    'works_remotely',
    'sought_mh_treatment',
    'has_mh_disorder_current',
    'comfortable_supervisor_mh_current',
    'mh_employer_consequences_current',
]

df[analysis_cols].isna().mean().mul(100).round(1)

age                                   0.0
gender                                0.2
works_remotely                        0.0
sought_mh_treatment                   0.0
has_mh_disorder_current               0.0
comfortable_supervisor_mh_current    20.0
mh_employer_consequences_current     20.0
dtype: float64

In [5]:
df.groupby('self_employed')['comfortable_supervisor_mh_current'].apply(lambda x: x.isna().mean() * 100)

self_employed
0      0.0
1    100.0
Name: comfortable_supervisor_mh_current, dtype: float64

## Age: real outliers, not just extreme values

Age was self-reported, and a couple of respondents mistyped and/or misrepresented their age.

In [6]:
df['age'].describe()

count    1433.000000
mean       34.286113
std        11.290931
min         3.000000
25%        28.000000
50%        33.000000
75%        39.000000
max       323.000000
Name: age, dtype: float64

The max is 323 and the min is 3, neither is a real working-age respondent. Rather than picking an 
arbitrary cutoff, I checked how many rows actually fall outside a realistic working-age range 
(18-75) before deciding anything.

Only 5 rows out of 1,433 (0.3%) fell outside that range: ages 3, 15, 17, 99, and 323. That's too small a group to be worth guessing a replacement value for, so I dropped them.

In [7]:
age = df['age']
below_18 = age[age < 18]
above_75 = age[age > 75]

print("Below 18:", sorted(below_18.tolist()))
print("Above 75:", sorted(above_75.tolist()))
print("Total rows affected:", len(below_18) + len(above_75), "out of", len(df))

Below 18: [3, 15, 17]
Above 75: [99, 323]
Total rows affected: 5 out of 1433


In [8]:
before = len(df)
df = df[(df['age'] >= 18) & (df['age'] <= 75)].copy()
after = len(df)

print(f"Rows before: {before}, after: {after}, dropped: {before - after}")

Rows before: 1433, after: 1428, dropped: 5


## Gender: standardizing 70 free-text responses

Gender was also free text, producing 70 unique raw answers for what should be a handful of categories: capitalization and spacing variants ('Female', 'female ', 'F', 'f'), typos ('mail', 'Malr'), and a smaller set of non-binary, genderqueer, or unclear responses.

In [9]:
for val in df['gender'].unique():
    print(val)

Male
male
Male 
Female
M
female
m
I identify as female.
female 
Bigender
non-binary
Female assigned at birth 
F
Woman
man
fm
f
Cis female 
Transitioned, M2F
Genderfluid (born female)
Other/Transfeminine
Female or Multi-Gender Femme
Female 
woman
female/woman
Cis male
Male.
Androgynous
male 9:1 female, roughly
nan
Male (cis)
nb masculine
Cisgender Female
Man
Sex is male
none of your business
genderqueer
cis male
Human
Genderfluid
Enby
Malr
genderqueer woman
mtf
Queer
Agender
Dude
Fluid
I'm a man why didn't you make this a drop down question. You should of asked sex? And I would of answered yes please. Seriously how much text can this take? 
mail
M|
Male/genderqueer
fem
Nonbinary
male 
human
Female (props for making this a freeform field, though)
 Female
Unicorn
Cis Male
Male (trans, FtM)
Cis-woman
Genderqueer
cisdude
Genderflux demi-girl
female-bodied; no feelings about gender
cis man
AFAB
Transgender woman
MALE


I split these into four categories instead of the usual two or three:
- **Male** / **Female**: standard variants, typos, and trans respondents classified by their 
  affirmed gender (e.g. "Transitioned, M2F" -> Female), not assumed birth sex 
- **Self-described / Other**: respondents who named an identity outside male/female (agender, genderfluid, 
  genderqueer, etc.), including single-word answers that still tell you something (not male, not female) even without detail
- **Not reported**: blank answers, plus responses that give zero usable information rather than 
  naming an identity ("none of your business", "Human")

I kept "Not reported" separate from "Self-described / Other" on purpose. Lumping a real stated identity in with a refusal to answer blurs two very different things.

In [10]:
import re

NONBINARY_KEYWORDS = [
    'agender', 'androgyn', 'bigender', 'enby', 'fluid', 'genderqueer',
    'nonbinary', 'non-binary', 'nb masculine', 'queer', 'unicorn', 'other', 'transfeminine',
]

NOT_REPORTED_KEYWORDS = [
    'none of your business', 'human'
]

def classify_gender(x):
    if pd.isna(x):
        return 'Not reported'
    s = str(x).strip().lower()

    for keyword in NOT_REPORTED_KEYWORDS:
        if keyword in s:
            return 'Not reported'

    for keyword in NONBINARY_KEYWORDS:
        if keyword in s:
            return 'Self-described / Other'

    if 'mtf' in s or 'm2f' in s or 'transgender woman' in s:
        return 'Female'

    if 'ftm' in s or 'f2m' in s:
        return 'Male'

    if s in ('m', 'mail', 'malr', 'm|'):
        return 'Male'

    if s in ('f', 'fm'):
        return 'Female'

    if 'female' in s or 'woman' in s or s.startswith('fem') or 'afab' in s:
        return 'Female'

    if 'male' in s or re.search(r'\bman\b', s) or 'dude' in s or 'sex is male' in s:
        return 'Male'

    return 'Self-described / Other'

In [11]:
df['gender_clean'] = df['gender'].apply(classify_gender)
df['gender_clean'].value_counts()

gender_clean
Male                      1054
Female                     345
Self-described / Other      23
Not reported                 6
Name: count, dtype: int64

Final counts: Male (1,054), Female (345), Self-described/Other (23), Not reported (6). Every one 
of the 1,428 original rows remaining after the age-outlier drop is accounted for.

In [12]:
df.loc[df['gender_clean'] == 'Not reported', 'gender'].unique()

<StringArray>
[nan, 'none of your business', 'Human', 'human']
Length: 4, dtype: str

In [13]:
df.to_csv('mental_health_in_tech_2016_cleaned.csv', index=False)
print("Saved. Final shape:", df.shape)

Saved. Final shape: (1428, 64)


## Summary of cleaning decisions

| Issue | Decision | Rows affected |
|---|---|---|
| High missingness in conditional columns | Left as NaN, reflects survey skip logic | N/A (structural) |
| Age outliers (3, 15, 17, 99, 323) | Dropped rows outside 18-75 | 5 of 1,433 (0.3%) |
| Gender free text (70 unique values, after age drop) | Standardized into 4 categories | 1,428 (all remaining rows recoded) |

Next steps: load `mental_health_in_tech_2016_cleaned.csv` into BigQuery, rebuild the four SQL queries against clean data, 
and rebuild the charts in Tableau instead of Google Sheets.